In [69]:
from pathlib import Path
import json
from collections import Counter


# ============================================================
# PROJECT PATHS
# ============================================================

ROOT = Path(r"C:\amrita_uni\s6\NLP\project\Rubric-based-evaluation-of-PL-SQL-code\Rubric-based-evaluation-of-PL-SQL-code")

SCHEMA_ARTIFACT_DIR = (
    ROOT / "artifacts" / "schema_output"
)

RAG_ARTIFACT_DIR = (
    ROOT / "artifacts" / "rag_output"
)

ARTIFACT_DIR = (
    ROOT / "artifacts" / "test_planning_output"
)

ARTIFACT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# LOAD ARTIFACTS
# ============================================================

schema = json.loads(
    (SCHEMA_ARTIFACT_DIR / "schema.json")
    .read_text(encoding="utf-8")
)

prompt_decomposition = json.loads(
    (SCHEMA_ARTIFACT_DIR / "prompt_decomposition.json")
    .read_text(encoding="utf-8")
)

retrieved_mutations = json.loads(
    (RAG_ARTIFACT_DIR / "retrieved_mutations.json")
    .read_text(encoding="utf-8")
)

rag_quality = json.loads(
    (RAG_ARTIFACT_DIR / "rag_quality.json")
    .read_text(encoding="utf-8")
)

print("Artifacts loaded.")
# ============================================================
# LOAD OPERATION ARTIFACTS
# ============================================================

operations = json.loads(
    (
        SCHEMA_ARTIFACT_DIR /
        "operations.json"
    ).read_text(
        encoding="utf-8"
    )
)

operation_behavior_map = json.loads(
    (
        SCHEMA_ARTIFACT_DIR /
        "operation_behavior_map.json"
    ).read_text(
        encoding="utf-8"
    )
)

print(
    "Operations loaded:",
    len(operations)
)

print(
    json.dumps(
        operation_behavior_map,
        indent=2
    )
)

Artifacts loaded.
Operations loaded: 3
{
  "classification_logic": "UpdateCustomerClass",
  "boundary_validation": "UpdateCustomerClass",
  "transaction_sensitive_operation": "SafeWithdrawal",
  "cross_table_dependency": "CloseBranch"
}


In [70]:
# ============================================================
# TESTCASE DISTRIBUTION POLICY
# ============================================================

TESTCASE_DISTRIBUTION_POLICY = {

    "normal_cases": 3,

    "edge_cases": 7
}


EDGE_CASE_PRIORITY = {

    "critical": 3,

    "high": 2,

    "medium": 1
}

print(json.dumps(
    TESTCASE_DISTRIBUTION_POLICY,
    indent=2
))

{
  "normal_cases": 3,
  "edge_cases": 7
}


In [71]:
# ============================================================
# TEST INTENT LIBRARY
# ============================================================

TEST_INTENT_LIBRARY = {

    "threshold_logic_failure": [

        "threshold_equal_validation",

        "threshold_minus_one_validation",

        "threshold_plus_one_validation"
    ],

    "rollback_failure": [

        "exact_balance_validation",

        "insufficient_balance_validation",

        "rollback_consistency_validation"
    ],

    "referential_integrity_failure": [

        "dependency_transfer_validation",

        "orphan_record_validation",

        "cross_table_consistency_validation"
    ],

    "exception_swallowing": [

        "invalid_input_validation",

        "exception_propagation_validation"
    ],

    "null_handling_failure": [

        "null_input_validation",

        "empty_result_validation"
    ]
}

print(
    json.dumps(
        TEST_INTENT_LIBRARY,
        indent=2
    )
)

{
  "threshold_logic_failure": [
    "threshold_equal_validation",
    "threshold_minus_one_validation",
    "threshold_plus_one_validation"
  ],
  "rollback_failure": [
    "exact_balance_validation",
    "insufficient_balance_validation",
    "rollback_consistency_validation"
  ],
  "referential_integrity_failure": [
    "dependency_transfer_validation",
    "orphan_record_validation",
    "cross_table_consistency_validation"
  ],
  "exception_swallowing": [
    "invalid_input_validation",
    "exception_propagation_validation"
  ],
  "null_handling_failure": [
    "null_input_validation",
    "empty_result_validation"
  ]
}


SECTION 4 — MUTATION → TESTCASE MAPPING

In [72]:
def build_mutation_testcase_mapping(
    retrieved_mutations,
    operation_behavior_map
):
    """
    Mutation
    ->
    Operation
    ->
    Test Intent
    """

    mappings = []

    for mutation in retrieved_mutations:

        semantic_category = (
            mutation[
                "semantic_category"
            ]
        )

        operation = (
            operation_behavior_map.get(
                semantic_category
            )
        )

        procedural_risk = (
            mutation[
                "procedural_risk"
            ]
        )

        mappings.append({

            "mutation_id":
                mutation["id"],

            "mutation_name":
                mutation[
                    "mutation_name"
                ],

            "risk_pattern":
                procedural_risk,

            "target_operation":
                operation,

            "test_intents":
                TEST_INTENT_LIBRARY.get(
                    procedural_risk,
                    []
                ),

            "behavioral_focus":
                mutation.get(
                    "testcase_strategy",
                    []
                ),

            "priority":
                mutation.get(
                    "testcase_priority",
                    "medium"
                )
        })

    return mappings


mutation_testcase_mapping = (
    build_mutation_testcase_mapping(
        retrieved_mutations,
        operation_behavior_map
    )
)

print(
    json.dumps(
        mutation_testcase_mapping[:2],
        indent=2
    )
)

[
  {
    "mutation_id": "MUT-044",
    "mutation_name": "ROLLBACK Removal",
    "risk_pattern": "rollback_failure",
    "target_operation": "SafeWithdrawal",
    "test_intents": [
      "exact_balance_validation",
      "insufficient_balance_validation",
      "rollback_consistency_validation"
    ],
    "behavioral_focus": [
      "transaction_consistency_test",
      "rollback_validation"
    ],
    "priority": "medium"
  },
  {
    "mutation_id": "MUT-045",
    "mutation_name": "Savepoint Mutation",
    "risk_pattern": "rollback_failure",
    "target_operation": "SafeWithdrawal",
    "test_intents": [
      "exact_balance_validation",
      "insufficient_balance_validation",
      "rollback_consistency_validation"
    ],
    "behavioral_focus": [
      "transaction_consistency_test",
      "rollback_validation"
    ],
    "priority": "medium"
  }
]


SECTION 5 — EDGE CASE PLANNING

In [73]:
def generate_edge_case_plan(
    mappings
):
    """
    Generate mutation-driven operation-centric
    edge case plans.
    """

    planned = []

    used_intents = set()

    for mapping in mappings:

        for intent in mapping[
            "test_intents"
        ]:

            if intent in used_intents:
                continue

            planned.append({

                "testcase_type":
                    "edge_case",

                "test_intent":
                    intent,

                "target_operations": [

                    mapping[
                        "target_operation"
                    ]

                ],

                "risk_pattern":
                    mapping[
                        "risk_pattern"
                    ],

                "mutation_reference":
                    mapping[
                        "mutation_id"
                    ],

                "behavioral_focus":
                    mapping[
                        "behavioral_focus"
                    ],

                "priority":
                    mapping[
                        "priority"
                    ]
            })

            used_intents.add(
                intent
            )

            if (
                len(planned)
                >=
                TESTCASE_DISTRIBUTION_POLICY[
                    "edge_cases"
                ]
            ):
                return planned

    return planned


edge_case_plan = (
    generate_edge_case_plan(
        mutation_testcase_mapping
    )
)

print(
    json.dumps(
        edge_case_plan[:5],
        indent=2
    )
)

[
  {
    "testcase_type": "edge_case",
    "test_intent": "exact_balance_validation",
    "target_operations": [
      "SafeWithdrawal"
    ],
    "risk_pattern": "rollback_failure",
    "mutation_reference": "MUT-044",
    "behavioral_focus": [
      "transaction_consistency_test",
      "rollback_validation"
    ],
    "priority": "medium"
  },
  {
    "testcase_type": "edge_case",
    "test_intent": "insufficient_balance_validation",
    "target_operations": [
      "SafeWithdrawal"
    ],
    "risk_pattern": "rollback_failure",
    "mutation_reference": "MUT-044",
    "behavioral_focus": [
      "transaction_consistency_test",
      "rollback_validation"
    ],
    "priority": "medium"
  },
  {
    "testcase_type": "edge_case",
    "test_intent": "rollback_consistency_validation",
    "target_operations": [
      "SafeWithdrawal"
    ],
    "risk_pattern": "rollback_failure",
    "mutation_reference": "MUT-044",
    "behavioral_focus": [
      "transaction_consistency_test",
     

SECTION 6 — NORMAL CASE PLANNING

In [74]:
def generate_normal_case_plan():

    planned = []

    for operation in operations:

        planned.append({

            "testcase_type":
                "normal_case",

            "test_intent":
                "successful_execution",

            "target_operations": [

                operation[
                    "operation_name"
                ]

            ],

            "priority":
                "medium"
        })

    return planned[
        :
        TESTCASE_DISTRIBUTION_POLICY[
            "normal_cases"
        ]
    ]


normal_case_plan = (
    generate_normal_case_plan()
)

print(
    json.dumps(
        normal_case_plan,
        indent=2
    )
)

[
  {
    "testcase_type": "normal_case",
    "test_intent": "successful_execution",
    "target_operations": [
      "UpdateCustomerClass"
    ],
    "priority": "medium"
  },
  {
    "testcase_type": "normal_case",
    "test_intent": "successful_execution",
    "target_operations": [
      "CloseBranch"
    ],
    "priority": "medium"
  },
  {
    "testcase_type": "normal_case",
    "test_intent": "successful_execution",
    "target_operations": [
      "SafeWithdrawal"
    ],
    "priority": "medium"
  }
]


In [75]:
combined_test_plan = (

    edge_case_plan

    +

    normal_case_plan
)

print("\nTotal Planned Testcases:",
      len(combined_test_plan))


Total Planned Testcases: 10


SECTION 8 — PROCEDURAL COVERAGE MATRIX

In [76]:
def build_coverage_matrix(
    combined_plan
):
    """
    Operation-centric coverage.
    """

    operation_coverage = Counter()

    testcase_distribution = Counter()

    for testcase in combined_plan:

        testcase_distribution[
            testcase[
                "testcase_type"
            ]
        ] += 1

        for operation in testcase.get(
            "target_operations",
            []
        ):

            operation_coverage[
                operation
            ] += 1

    return {

        "operation_coverage":
            dict(
                operation_coverage
            ),

        "testcase_distribution":
            dict(
                testcase_distribution
            ),

        "total_testcases":
            len(
                combined_plan
            )
    }


coverage_matrix = (
    build_coverage_matrix(
        combined_test_plan
    )
)

print(
    json.dumps(
        coverage_matrix,
        indent=2
    )
)

{
  "operation_coverage": {
    "SafeWithdrawal": 4,
    "UpdateCustomerClass": 4,
    "CloseBranch": 2
  },
  "testcase_distribution": {
    "edge_case": 7,
    "normal_case": 3
  },
  "total_testcases": 10
}


In [77]:
def build_explainability_links(
    combined_plan
):
    """
    Explainability mapping.
    """

    explainability = []

    for testcase in combined_plan:

        explainability.append({

            "test_intent":
                testcase.get(
                    "test_intent"
                ),

            "target_operations":
                testcase.get(
                    "target_operations"
                ),

            "risk_pattern":
                testcase.get(
                    "risk_pattern"
                ),

            "feedback_strategy":
                "behavioral_failure_analysis"
        })

    return explainability


explainability_links = (
    build_explainability_links(
        combined_test_plan
    )
)

print(
    json.dumps(
        explainability_links[:5],
        indent=2
    )
)

[
  {
    "test_intent": "exact_balance_validation",
    "target_operations": [
      "SafeWithdrawal"
    ],
    "risk_pattern": "rollback_failure",
    "feedback_strategy": "behavioral_failure_analysis"
  },
  {
    "test_intent": "insufficient_balance_validation",
    "target_operations": [
      "SafeWithdrawal"
    ],
    "risk_pattern": "rollback_failure",
    "feedback_strategy": "behavioral_failure_analysis"
  },
  {
    "test_intent": "rollback_consistency_validation",
    "target_operations": [
      "SafeWithdrawal"
    ],
    "risk_pattern": "rollback_failure",
    "feedback_strategy": "behavioral_failure_analysis"
  },
  {
    "test_intent": "threshold_equal_validation",
    "target_operations": [
      "UpdateCustomerClass"
    ],
    "risk_pattern": "threshold_logic_failure",
    "feedback_strategy": "behavioral_failure_analysis"
  },
  {
    "test_intent": "threshold_minus_one_validation",
    "target_operations": [
      "UpdateCustomerClass"
    ],
    "risk_pattern":

SECTION 10 — EXPORT TESTCASE PLANNING ARTIFACTS

In [78]:
(ARTIFACT_DIR / "combined_test_plan.json").write_text(
    json.dumps(
        combined_test_plan,
        indent=2
    ),
    encoding="utf-8"
)

(ARTIFACT_DIR / "coverage_matrix.json").write_text(
    json.dumps(
        coverage_matrix,
        indent=2
    ),
    encoding="utf-8"
)

(ARTIFACT_DIR / "mutation_testcase_mapping.json").write_text(
    json.dumps(
        mutation_testcase_mapping,
        indent=2
    ),
    encoding="utf-8"
)

(ARTIFACT_DIR / "explainability_links.json").write_text(
    json.dumps(
        explainability_links,
        indent=2
    ),
    encoding="utf-8"
)

print("Test planning artifacts exported.")

Test planning artifacts exported.


In [79]:
assert len(combined_test_plan) == 10

assert (
    coverage_matrix["testcase_distribution"]
    ["edge_case"]
    >= 7
)

assert (
    coverage_matrix["testcase_distribution"]
    ["normal_case"]
    >= 3
)

assert (
    len(
        coverage_matrix[
            "operation_coverage"
        ]
    )
    >= 2
)

assert len(explainability_links) == 10

print("Semantic testcase planning tests passed.")

Semantic testcase planning tests passed.


In [80]:
print(
    json.dumps(
        combined_test_plan[:5],
        indent=2
    )
)

[
  {
    "testcase_type": "edge_case",
    "test_intent": "exact_balance_validation",
    "target_operations": [
      "SafeWithdrawal"
    ],
    "risk_pattern": "rollback_failure",
    "mutation_reference": "MUT-044",
    "behavioral_focus": [
      "transaction_consistency_test",
      "rollback_validation"
    ],
    "priority": "medium"
  },
  {
    "testcase_type": "edge_case",
    "test_intent": "insufficient_balance_validation",
    "target_operations": [
      "SafeWithdrawal"
    ],
    "risk_pattern": "rollback_failure",
    "mutation_reference": "MUT-044",
    "behavioral_focus": [
      "transaction_consistency_test",
      "rollback_validation"
    ],
    "priority": "medium"
  },
  {
    "testcase_type": "edge_case",
    "test_intent": "rollback_consistency_validation",
    "target_operations": [
      "SafeWithdrawal"
    ],
    "risk_pattern": "rollback_failure",
    "mutation_reference": "MUT-044",
    "behavioral_focus": [
      "transaction_consistency_test",
     

Purpose
Behavior-aware testcase planning

Converts:

retrieved semantic mutations
→ structured testcase plan
OUTPUT FOLDER
artifacts/test_planning_output/
1. combined_test_plan.json

MOST IMPORTANT planning artifact.

Contains all 10 planned testcases.

Example:

{
  "template":
     "threshold_minus_one_case",

  "semantic_category":
     "boundary_validation"
}

This drives:

testcase generation
execution
scoring

2. coverage_matrix.json

Tracks:

testcase diversity
semantic coverage

Example:

{
  "boundary_validation":3,
  "transaction_sensitive_operation":4
}

VERY important for explainability.

3. mutation_testcase_mapping.json

Maps:

mutation
    ↓
testcase strategy

VERY important later.

4. explainability_links.json

This is HUGE.

This enables:

failed testcase
    ↓
behavioral explanation

This becomes the core explainability layer later.